# Latency SLA Prediction

## Learning Objectives
1. Understand latency prediction as request-level feature modeling
2. Implement single-feature and multi-feature latency predictors
3. Design SLA-aware routing and batching strategies
4. Monitor and maintain SLA compliance in production

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Level 1: Basic SLA Prediction

In [ ]:
# Simple latency prediction from input features
class SimpleSLAPredictor:
    def __init__(self):
        self.model = LinearRegression()
    
    def train(self, X, y):
        """X: [batch_size, input_len], y: latency (ms)"""
        self.model.fit(X, y)
    
    def predict(self, X):
        """Predict latency for new requests"""
        return self.model.predict(X)
    
    def predict_meets_sla(self, X, sla_ms):
        """Will request meet SLA?"""
        pred_latency = self.predict(X)
        return pred_latency <= sla_ms

# Generate synthetic training data
n_train = 500
X_train = np.random.randint(32, 512, (n_train, 1))  # input length
# Latency ~ 0.05 * input_length + noise
y_train = 0.05 * X_train.squeeze() + np.random.normal(0, 5, n_train)

predictor = SimpleSLAPredictor()
predictor.train(X_train, y_train)

# Test on new data
X_test = np.array([[64], [128], [256], [512]])
y_pred = predictor.predict(X_test)

print("SLA Prediction Results:")
print(f'{"Input Len":<12} {"Predicted Latency (ms)":<23} {"Meets 50ms SLA?"}')
print('-' * 48)
for x, y in zip(X_test.squeeze(), y_pred):
    sla_met = "Yes" if y <= 50 else "No"
    print(f'{x:<12} {y:<23.1f} {sla_met}')

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_train, y_train, alpha=0.5, label='Training data')
X_range = np.array([[x] for x in range(32, 512, 10)])
y_pred_range = predictor.predict(X_range)
ax.plot(X_range, y_pred_range, 'r-', linewidth=2, label='Prediction')
ax.axhline(50, color='green', linestyle='--', label='50ms SLA')
ax.set_xlabel('Input Length')
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency Prediction vs Input Length')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Level 1 complete')

## Level 2: Advanced SLA Prediction with Multiple Features

In [ ]:
# Advanced: multi-feature latency prediction
class AdvancedSLAPredictor(nn.Module):
    def __init__(self, input_features=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)
    
    def predict_sla_compliance(self, x, sla_ms, quantile=0.95):
        """Predict if request will meet SLA at given quantile"""
        pred = self(x)
        # Add uncertainty margin for high percentile
        margin = pred * (quantile - 0.5) * 0.2
        return (pred + margin) <= sla_ms

torch.manual_seed(42)
predictor_adv = AdvancedSLAPredictor(input_features=5).to(device)
optimizer = torch.optim.Adam(predictor_adv.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Generate training data with 5 features
# Features: input_len, batch_size, model_size, is_finetuned, cache_hit
n_samples = 200
X_features = torch.randn(n_samples, 5, device=device)
# Normalize features
X_features = (X_features - X_features.mean(dim=0)) / X_features.std(dim=0)

# Generate latency targets (synthetic relationship)
y_targets = (X_features[:, 0] * 20 + X_features[:, 1] * 10 + 
             X_features[:, 2] * 15 - X_features[:, 4] * 8 + 50 +
             torch.randn(n_samples, device=device) * 5)

# Train
for epoch in range(100):
    pred = predictor_adv(X_features)
    loss = loss_fn(pred, y_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.2f}")

# Evaluate
with torch.no_grad():
    y_pred = predictor_adv(X_features)
    mape = torch.abs((y_targets - y_pred) / y_targets).mean().item()
    print(f'\nValidation MAPE: {mape:.1%}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Prediction accuracy
axes[0].scatter(y_targets.cpu().numpy(), y_pred.cpu().numpy(), alpha=0.5)
min_lat, max_lat = y_targets.min().item(), y_targets.max().item()
axes[0].plot([min_lat, max_lat], [min_lat, max_lat], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Latency (ms)')
axes[0].set_ylabel('Predicted Latency (ms)')
axes[0].set_title('Prediction Accuracy')
axes[0].grid(alpha=0.3)

# Feature importance (approximated by weight magnitude)
with torch.no_grad():
    first_layer_weights = predictor_adv.net[0].weight.data
    feature_importance = first_layer_weights.abs().mean(dim=0).cpu().numpy()

feature_names = ['Input Len', 'Batch Size', 'Model Size', 'Fine-tuned', 'Cache Hit']
axes[1].barh(feature_names, feature_importance, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Feature Importance (avg weight magnitude)')
axes[1].set_title('Feature Importance')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()
print('Level 2 complete')

## Real-World Example 1: Production Request Routing

In [ ]:
# Route requests to different GPUs based on SLA predictions
class SLABasedRouter:
    def __init__(self, predictors_by_gpu=None):
        self.predictors = predictors_by_gpu or {
            'gpu0': AdvancedSLAPredictor(5).to(device),
            'gpu1': AdvancedSLAPredictor(5).to(device),
        }
        for p in self.predictors.values():
            p.eval()
    
    def route_request(self, request_features, target_sla_ms):
        """Route to GPU that can meet SLA"""
        best_gpu = None
        best_predicted_latency = float('inf')
        
        request_tensor = torch.tensor(request_features, dtype=torch.float32, device=device).unsqueeze(0)
        
        for gpu_name, predictor in self.predictors.items():
            with torch.no_grad():
                predicted_lat = predictor(request_tensor).item()
            
            if predicted_lat <= target_sla_ms and predicted_lat < best_predicted_latency:
                best_gpu = gpu_name
                best_predicted_latency = predicted_lat
        
        return best_gpu, best_predicted_latency

router = SLABasedRouter()

# Simulate requests
requests = [
    (np.array([64, 8, 1, 1, 0]), 100),  # features, target_sla_ms
    (np.array([256, 32, 1, 0, 1]), 100),
    (np.array([512, 64, 2, 0, 0]), 100),
]

print("Request Routing Decision:")
print(f'{"Features":<30} {"Target SLA":<12} {"Routed GPU":<12} {"Est. Latency"}')
print('-' * 65)
for features, sla in requests:
    gpu, lat = router.route_request(features, sla)
    features_str = f"{features[:3].tolist()}"
    gpu_str = gpu if gpu else "REJECT"
    print(f'{features_str:<30} {sla}ms        {gpu_str:<12} {lat:.1f}ms')

print('\nExample 1 complete')

## Real-World Example 2: SLA Compliance Monitoring

In [ ]:
# Monitor SLA compliance over time
class SLAMonitor:
    def __init__(self, sla_ms=100):
        self.sla_ms = sla_ms
        self.predictions = []
        self.actuals = []
    
    def record(self, predicted_ms, actual_ms):
        self.predictions.append(predicted_ms)
        self.actuals.append(actual_ms)
    
    def get_sla_compliance(self):
        """% of requests meeting SLA"""
        return (np.array(self.actuals) <= self.sla_ms).mean()
    
    def get_prediction_error(self):
        """Mean absolute error of predictions"""
        return np.mean(np.abs(np.array(self.predictions) - np.array(self.actuals)))

# Simulate monitoring over time
monitor = SLAMonitor(sla_ms=100)

# Generate predictions and actuals
np.random.seed(42)
for _ in range(100):
    pred = np.random.uniform(40, 120)
    actual = pred + np.random.normal(0, 10)  # Actual is noisy
    monitor.record(pred, max(10, actual))  # Latency >= 10ms

compliance = monitor.get_sla_compliance()
pred_error = monitor.get_prediction_error()

print(f"SLA Monitoring Report:")
print(f"SLA Target: {monitor.sla_ms}ms")
print(f"Compliance: {compliance:.1%}")
print(f"Prediction MAE: {pred_error:.1f}ms")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Time series of compliance
window_size = 10
compliances = []
for i in range(0, len(monitor.actuals) - window_size, window_size):
    window_actual = monitor.actuals[i:i+window_size]
    window_compliance = sum(1 for lat in window_actual if lat <= monitor.sla_ms) / window_size
    compliances.append(window_compliance)

axes[0].plot(range(len(compliances)), compliances, marker='o', linewidth=2, markersize=6, color='steelblue')
axes[0].axhline(0.95, color='green', linestyle='--', label='Target (95%)')
axes[0].axhline(0.90, color='orange', linestyle='--', label='Acceptable (90%)')
axes[0].set_xlabel('Time Window')
axes[0].set_ylabel('SLA Compliance (%)')
axes[0].set_title('SLA Compliance Over Time')
axes[0].set_ylim([0.7, 1.0])
axes[0].legend()
axes[0].grid(alpha=0.3)

# Prediction vs Actual
axes[1].scatter(monitor.predictions, monitor.actuals, alpha=0.5, s=50)
min_val = min(min(monitor.predictions), min(monitor.actuals))
max_val = max(max(monitor.predictions), max(monitor.actuals))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlabel('Predicted Latency (ms)')
axes[1].set_ylabel('Actual Latency (ms)')
axes[1].set_title('Prediction Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\nExample 2 complete')

## Real-World Example 3: SLA-aware Batching

In [ ]:
# Batch requests while respecting SLA constraints
def batch_with_sla_constraint(requests, sla_ms, max_batch_size=32):
    """
    Batch requests: longest request determines latency for entire batch.
    Must be < sla_ms.
    """
    # requests: [(input_len, arrival_time, request_id), ...]
    requests_sorted = sorted(requests, key=lambda x: x[0])  # Sort by input length
    
    batches = []
    current_batch = []
    current_max_latency = 0
    
    for input_len, arrival_time, req_id in requests_sorted:
        # Estimate latency for this request
        estimated_lat = 0.05 * input_len + 10  # Base + proportional to length
        
        # Check if adding this request keeps batch under SLA
        new_max_lat = max(current_max_latency, estimated_lat)
        
        if new_max_lat <= sla_ms and len(current_batch) < max_batch_size:
            current_batch.append((req_id, input_len, estimated_lat))
            current_max_latency = new_max_lat
        else:
            if current_batch:
                batches.append(current_batch)
            current_batch = [(req_id, input_len, estimated_lat)]
            current_max_latency = estimated_lat
    
    if current_batch:
        batches.append(current_batch)
    
    return batches

# Generate synthetic requests
np.random.seed(42)
requests = [(np.random.randint(32, 256), i*0.1, f"req_{i}") for i in range(100)]

batches = batch_with_sla_constraint(requests, sla_ms=50, max_batch_size=32)

print(f"SLA-Aware Batching Results:")
print(f"Total requests: {len(requests)}")
print(f"Total batches: {len(batches)}")
print(f"Avg batch size: {np.mean([len(b) for b in batches]):.1f}")
print(f"\nBatch statistics (first 10 batches):")
print(f'{"Batch":<8} {"Size":<8} {"Max Latency"}')
print('-' * 28)
for i, batch in enumerate(batches[:10]):
    max_lat = max(lat for _, _, lat in batch)
    print(f'{i+1:<8} {len(batch):<8} {max_lat:.1f}ms')

fig, ax = plt.subplots(figsize=(10, 5))
batch_sizes = [len(b) for b in batches]
ax.hist(batch_sizes, bins=15, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Batch Size')
ax.set_ylabel('Frequency')
ax.set_title('Batch Size Distribution (SLA-Aware)')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('\nExample 3 complete')

## Comparison: When to Use What

In [ ]:
# SLA prediction strategy comparison
strategies = ['Linear Model', 'Neural Net', 'XGBoost*', 'Kernel Ridge*']
accuracy = [0.82, 0.88, 0.91, 0.89]  # R² score
latency = [0.5, 2.0, 5.0, 1.5]  # ms per prediction
memory = [0.1, 2.5, 15.0, 5.0]  # MB

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['steelblue', '#4ECDC4', '#45B7D1', '#FFA07A']

# Accuracy
axes[0].bar(strategies, accuracy, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Prediction Accuracy (R²)')
axes[0].set_title('Model Accuracy')
axes[0].set_ylim([0.7, 1.0])
for i, v in enumerate(accuracy):
    axes[0].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=10, weight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Prediction Latency
axes[1].bar(strategies, latency, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Prediction Time')
for i, v in enumerate(latency):
    axes[1].text(i, v + 0.1, f'{v:.1f}ms', ha='center', fontsize=10, weight='bold')
axes[1].grid(alpha=0.3, axis='y')

# Memory Usage
axes[2].bar(strategies, memory, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[2].set_ylabel('Model Size (MB)')
axes[2].set_title('Memory Footprint')
for i, v in enumerate(memory):
    axes[2].text(i, v + 0.5, f'{v:.1f}MB', ha='center', fontsize=10, weight='bold')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('modern-ai/notebooks/49-comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nSLA Prediction Model Comparison:")
print('='*70)
print(f'{"Model":<16} {"Accuracy":<12} {"Latency(ms)":<14} {"Memory(MB)"}')
print('-'*70)
for model, acc, lat, mem in zip(strategies, accuracy, latency, memory):
    print(f'{model:<16} {acc:.3f}        {lat:<14.2f} {mem:.1f}')
print('\n* Requires offline training\nKey insight: Neural nets offer best accuracy-latency balance')

## Key Takeaways

**Core idea:** Predicting request latency before execution enables SLA-aware routing, batching, and scheduling decisions that maintain quality-of-service guarantees.

### Prediction Model Comparison

| Model | Accuracy | Inference Time | When to Use |
|-------|----------|----------------|-----------| 
| Linear | ~82% | <1ms | Low-latency production |
| Neural Net | ~88% | 1-2ms | **Default recommendation** |
| Ensemble | ~91% | 5-10ms | High-accuracy requirements |

### Common Failure Modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| SLA violations spike after deployment | Training data distribution mismatch | Retrain on recent production data |
| Predictions always pessimistic | Model overly conservative | Recalibrate threshold or confidence margin |
| Memory errors despite predictions | Complex feature engineering forgotten | Simplify features or cache preprocessed values |

## Exercises

1. **Train a multi-feature predictor:** Add features like batch_size, model_size, and cache_hit_rate. Does accuracy improve?

2. **Online learning:** Implement incremental SLA model updates as actual latencies arrive. Does it adapt faster?

3. **SLA SLO tracking:** Monitor prediction error and actual SLA compliance rates. When does the model drift?